In [ ]:
# Import tkinter for GUI and ttk for improved widgets
import tkinter as tk# GUI
from tkinter import ttk, messagebox# pop up that di

# Define the RestaurantOrderManagementApp class which will encapsulate all the GUI and logic
class RestaurantOrderManagement:
    # Initialize the application by setting up the main window and widgets
    def __init__(self, root):
        self.root = root  # The main Tkinter window (root widget)
        self.root.title("Restaurant Management App")  # Set the title of the application window

        # A dictionary to store the menu items and their base prices in USD
        self.menu_items = {
            "FRIES MEAL": 2,
            "LUNCH MEAL": 2,
            "BURGER MEAL": 3,
            "PIZZA MEAL": 4,
            "CHEESE BURGER": 2.5,
            "DRINKS": 1
        }

        self.exchange_rate = 82  # Fixed exchange rate for currency conversion (e.g., 1 USD = 82 INR)

        # Call a method to set up the background image for the application
        # Note: This line will cause an error if 'background.png' is not present in the environment
        self.setup_background(root)

        # Create a frame widget to organize and contain other GUI elements
        frame = ttk.Frame(root)
        # Place the frame in the center of the root window
        frame.place(relx=0.5, rely=0.5, anchor=tk.CENTER)

        # Create a heading label for the application
        ttk.Label(
            frame,
            text="Restaurant Order Management",
            font=("Arial", 20, "bold")
        ).grid(row=0, columnspan=3, padx=10, pady=10)

        self.menu_labels = {}       # Dictionary to store references to menu item labels (e.g., "FRIES MEAL ($2):")
        self.menu_quantities = {}   # Dictionary to store references to quantity entry widgets for each menu item

        # Loop through each menu item to create a label and an entry widget for its quantity
        for i, (item, price) in enumerate(self.menu_items.items(), start=1):
            # Create a label for the current menu item, displaying its name and price
            label = ttk.Label(
                frame,
                text=f"{item} (${price}):", # Initial display with USD price
                font=("Arial", 12)
            )
            # Place the label in the grid layout
            label.grid(row=i, column=0, padx=10, pady=5)
            self.menu_labels[item] = label # Store the label reference

            # Create an entry widget for the user to input the quantity of the item
            quantity_entry = ttk.Entry(frame, width=5)
            # Place the entry widget next to its label
            quantity_entry.grid(row=i, column=1, padx=10, pady=5)
            self.menu_quantities[item] = quantity_entry # Store the entry widget reference

        # Variable to hold the currently selected currency (USD or INR)
        self.currency_var = tk.StringVar()
        # Create a label for currency selection
        ttk.Label(
            frame,
            text="Currency:",
            font=("Arial", 12)
        ).grid(
            row=len(self.menu_items) + 1, # Position after the last menu item
            column=0,
            padx=10,
            pady=5
        )

        # Create a dropdown (Combobox) for currency selection
        currency_dropdown = ttk.Combobox(
            frame,
            textvariable=self.currency_var, # Link to self.currency_var
            state="readonly", # Make it non-editable, only selectable from values
            width=18,
            values=("USD", "INR") # Available currency options
        )
        # Place the dropdown in the grid
        currency_dropdown.grid(
            row=len(self.menu_items) + 1,
            column=1,
            padx=10,
            pady=5
        )
        currency_dropdown.current(0)  # Set "USD" as the default selected currency
        # Bind the update_menu_prices method to changes in the currency selection
        self.currency_var.trace("w", self.update_menu_prices)

        # Create a button to trigger the order placement process
        order_button = ttk.Button(
            frame,
            text="Place Order",
            command=self.place_order # Call place_order method when button is clicked
        )
        # Place the order button in the grid, spanning all three columns
        order_button.grid(
            row=len(self.menu_items) + 2,
            columnspan=3,
            padx=10,
            pady=10
        )

    # Method to set up the background image on the main window
    def setup_background(self, root):
        bg_width, bg_height = 800, 600 # Define desired background image dimensions
        canvas = tk.Canvas(root, width=bg_width, height=bg_height) # Create a canvas for the background
        canvas.pack() # Pack the canvas to fill the window

        try:
            original_image = tk.PhotoImage(file="background.png") # Load the background image
            # Resize the image to fit the canvas dimensions (subsample for efficiency)
            background_image = original_image.subsample(
                original_image.width() // bg_width,
                original_image.height() // bg_height
            )

            # Place the image on the canvas at the top-left corner
            canvas.create_image(0, 0, anchor=tk.NW, image=background_image)
            canvas.image = background_image # Keep a reference to prevent garbage collection
        except tk.TclError:
            # Handle the case where the background.png file is not found
            print("Warning: background.png not found. Background image will not be displayed.")
            # Optionally, set a default background color if image is missing
            canvas.config(bg="lightgray")

    # Method to update the displayed menu prices based on the selected currency
    def update_menu_prices(self, *args):
        currency = self.currency_var.get() # Get the selected currency
        symbol = "₹" if currency == "INR" else "$" # Determine currency symbol
        rate = self.exchange_rate if currency == "INR" else 1 # Determine exchange rate

        # Iterate through each menu item and update its displayed price
        for item, label in self.menu_labels.items():
            price = self.menu_items[item] * rate # Calculate price in the selected currency
            # Update the label text with the new price and symbol
            label.config(text=f"{item} ({symbol}{price}):")

    # Method to process the order when the "Place Order" button is clicked
    def place_order(self):
        total_cost = 0 # Initialize total cost of the order
        order_summary = "Order Summary:\n" # Initialize order summary string
        currency = self.currency_var.get() # Get the selected currency
        symbol = "₹" if currency == "INR" else "$" # Determine currency symbol
        rate = self.exchange_rate if currency == "INR" else 1 # Determine exchange rate

        # Iterate through each menu item and its corresponding quantity entry widget
        for item, entry in self.menu_quantities.items():
            quantity_str = entry.get() # Get the quantity as a string
            # Check if the entered quantity is a valid digit
            if quantity_str.isdigit():
                quantity = int(quantity_str) # Convert quantity to integer
                price = self.menu_items[item] * rate # Calculate the item price in the selected currency
                cost = quantity * price # Calculate the cost for this item
                total_cost += cost # Add to the overall total cost

                # If the quantity is greater than zero, add it to the order summary
                if quantity > 0:
                    order_summary += (
                        f"{item}: {quantity} x {symbol}{price:.2f} = {symbol}{cost:.2f}\n" # Format to 2 decimal places
                    )
            else:
                # If quantity is not a digit, display an error and clear the invalid input
                messagebox.showerror("Invalid Input", f"Please enter a valid number for {item}.")
                entry.delete(0, tk.END) # Clear the invalid entry
                return # Stop processing the order

        # Check if any items were ordered (total_cost > 0)
        if total_cost > 0:
            order_summary += f"\nTotal Cost: {symbol}{total_cost:.2f}" # Add total cost to summary
            # Display the order summary in an information message box
            messagebox.showinfo("Order Placed", order_summary)
        else:
            # If no items were ordered, display an error message
            messagebox.showerror("Error", "Please order at least one item.")

# Main block to run the application when the script is executed
if __name__ == "__main__":
    root = tk.Tk() # Create the main Tkinter window
    app = RestaurantOrderManagement(root) # Create an instance of the RestaurantOrderManagement application
    root.geometry("800x600") # Set the initial size of the application window
    root.mainloop() # Start the Tkinter event loop, which waits for user interactions